In [43]:
library(dplyr)
library(tidyr)
library(stringr)
library(tibble)
library(pheatmap)
library(RColorBrewer)

step <- function(msg) {
    cat(sprintf("\n[%s] %s\n", format(Sys.time(), "%H:%M:%S"), msg))
    flush.console()
}

In [44]:
#! Acer & Amel
# dataset.A <- "Acer"
# col.A <- "leiden_res0.50"
# dataset.B <- "Amel"
# col.B <- "cell_type_cluster"

# oup_name <- paste0(dataset.A, "-", col.A, "_vs_", dataset.B, "-", col.B)
# auroc_file <- file.path("./metadata/MN_R_auroc", oup_name, "auroc.tsv")

# out_dir <- dirname(auroc_file)
# stopifnot(file.exists(auroc_file))
# dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

#! Acer & Hsal
# dataset.A <- "Acer"
# col.A <- "leiden_res0.50"
# dataset.B <- "Hsal"
# col.B <- "cluster_name"

# oup_name <- paste0(dataset.A, "-", col.A, "_vs_", dataset.B, "-", col.B)
# auroc_file <- file.path("./metadata/MN_R_auroc", oup_name, "auroc.tsv")

# out_dir <- dirname(auroc_file)
# stopifnot(file.exists(auroc_file))
# dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

#! Acer & Dmel
dataset.A <- "Acer"
col.A <- "leiden_res0.50"
dataset.B <- "Dmel"
col.B <- "afca_annotation"

oup_name <- paste0(dataset.A, "-", col.A, "_vs_", dataset.B, "-", col.B)
auroc_file <- file.path("./metadata/MN_R_auroc", oup_name, "auroc.tsv")

out_dir <- dirname(auroc_file)
stopifnot(file.exists(auroc_file))
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

In [45]:
# ---- 1. read auroc table ---------------------------------------------------
step("1. read auroc.tsv")
res <- read.delim(auroc_file)
stopifnot(all(c("Var1", "Var2", "AUROCs", "AUROCp") %in% colnames(res)))
cat("auroc rows:", nrow(res), "\n")


[20:58:59] 1. read auroc.tsv
auroc rows: 13924 


In [46]:
# ---- 3. keep cross-species pairs and add type labels ----------------------
step("3. keep cross-species pairs and attach types")
df <- res %>%
    filter(str_starts(Var1, paste0(dataset.B, "\\.")),
            str_starts(Var2, paste0(dataset.A, "\\."))) %>%
    separate(Var1, c("dataset.B", "cluster.B"), "\\.", extra = "merge") %>%
    separate(Var2, c("dataset.A", "cluster.A"), "\\.", extra = "merge") %>%
    mutate(
        Var1 = paste0(dataset.B, ".", cluster.B),
        Var2 = paste0(dataset.A, ".", cluster.A)
    )


[20:58:59] 3. keep cross-species pairs and attach types


In [47]:
# res %>%
#     filter(str_starts(Var1, paste0(dataset.B, "\\.")),
#             str_starts(Var2, paste0(dataset.A, "\\.")))

# res %>%
#     filter(str_starts(Var1, paste0(dataset.B, "\\.")),
#             str_starts(Var2, paste0(dataset.A, "\\."))) %>%
#     separate(Var1, c("dataset.B", "cluster.B"), "\\.", extra = "merge")

In [48]:
stopifnot(
    nrow(df) > 0,
    !anyNA(df$cluster.B),
    !anyNA(df$cluster.A)
)
cat("cross-species pairs:", nrow(df),
    "(", length(unique(df$Var2)), "x", length(unique(df$Var1)), ")\n")

anno_file <- file.path(out_dir, "auroc.anno.tsv")
write.table(df, anno_file, sep = "\t", quote = FALSE, row.names = FALSE)
cat("   saved:", anno_file, "\n")

cross-species pairs: 3192 ( 42 x 76 )
   saved: ./metadata/MN_R_auroc/Acer-leiden_res0.50_vs_Dmel-afca_annotation/auroc.anno.tsv 


In [49]:
# ---- 4. wide matrix -------------------------------------------------------
step("4. build AUROC matrix")
auroc_s <- df %>%
    select(Var1, Var2, AUROCs) %>%
    spread(key = Var1, value = AUROCs) %>%
    column_to_rownames(var = "Var2") %>%
    as.matrix()

auroc_p <- df %>%
    select(Var1, Var2, AUROCp) %>%
    spread(key = Var1, value = AUROCp) %>%
    column_to_rownames(var = "Var2") %>%
    as.matrix()
stopifnot(!anyNA(auroc_s))
stopifnot(!anyNA(auroc_p))
cat("spearman matrix dim:", paste(dim(auroc_s), collapse = " x "), "\n")
cat("pearson matrix dim:", paste(dim(auroc_p), collapse = " x "), "\n")


[20:58:59] 4. build AUROC matrix
spearman matrix dim: 42 x 76 
pearson matrix dim: 42 x 76 


In [50]:
# ---- 5. hierarchical clustering (same as reference) -----------------------
step("5. cluster rows and columns")
hc.row <- hclust(dist(auroc_s))
auroc_s <- auroc_s[hc.row$order, ]
hc.col <- hclust(dist(t(auroc_s)))
auroc_s <- auroc_s[, hc.col$order]

hc.row <- hclust(dist(auroc_p))
auroc_p <- auroc_p[hc.row$order, ]
hc.col <- hclust(dist(t(auroc_p)))
auroc_p <- auroc_p[, hc.col$order]


[20:58:59] 5. cluster rows and columns


In [51]:
# ---- 6. number labels -----------------------------------------------------
text_s <- matrix("", nrow(auroc_s), ncol(auroc_s), dimnames = dimnames(auroc_s))
show <- auroc_s >= 0.5
text_s[show] <- as.character(signif(auroc_s[show], 2))

text_p <- matrix("", nrow(auroc_p), ncol(auroc_p), dimnames = dimnames(auroc_p))
show <- auroc_p >= 0.5
text_p[show] <- as.character(signif(auroc_p[show], 2))

In [52]:
# ---- 7. pheatmap ----------------------------------------------------------
step("6. draw pheatmap")
color.up.x   <- brewer.pal(6, "Reds")
color.down.x <- brewer.pal(6, "Blues")
color.use <- colorRampPalette(c(rev(color.down.x), "white", color.up.x))(50)

pdf_file_s <- file.path(out_dir, "auroc_s.pdf")
pheatmap(
    auroc_s,
    cluster_cols = TRUE,
    cluster_rows = TRUE,
    color = color.use,
    na_col = "white",
    cellwidth = 17,
    cellheight = 17,
    fontsize = 8,
    fontsize_row = 6,
    fontsize_col = 6,
    display_numbers = text_s,
    number_color = "black",
    show_rownames = TRUE,
    show_colnames = TRUE,
    angle_col = "45",
    main = paste0(dataset.B, " (row) vs ", dataset.A, " (col) [Spearman]"),
    filename = pdf_file_s
)

pdf_file_p <- file.path(out_dir, "auroc_p.pdf")
pheatmap(
    auroc_p,
    cluster_cols = TRUE,
    cluster_rows = TRUE,
    color = color.use,
    na_col = "white",
    cellwidth = 17,
    cellheight = 17,
    fontsize = 8,
    fontsize_row = 6,
    fontsize_col = 6,
    display_numbers = text_p,
    number_color = "black",
    show_rownames = TRUE,
    show_colnames = TRUE,
    angle_col = "45",
    main = paste0(dataset.B, " (row) vs ", dataset.A, " (col) [Pearson]"),
    filename = pdf_file_p
)
cat("saved:", pdf_file_p, "\n")
step("done")


[20:58:59] 6. draw pheatmap


saved: ./metadata/MN_R_auroc/Acer-leiden_res0.50_vs_Dmel-afca_annotation/auroc_p.pdf 

[20:58:59] done
